In [1]:
from useful_functions import format_datetime64, get_daterange_str

import numpy as np
import pandas as pd
import xarray as xr
import os
from datetime import datetime

data_out_dir = '../data_out/'

## Script settings

In [2]:
rhi_angle = 25

## Load data

In [3]:
ds_icon_in = xr.open_dataset(f'/Users/geoalxx/Python/glacier_space/data_icon_h3/icon_1415_{rhi_angle}.nc')
ds_rhi_in = xr.open_dataset(f'/Users/geoalxx/Python/glacier_space/data_obs_h3/lidar/combined_rhi_period_1_6_17.nc')

In [5]:
range_gates = ds_rhi_in.attrs['range']
config_str = ds_rhi_in.attrs['File_Configuration']
config_dict = dict(line.split("=", 1) for line in config_str.splitlines())

print('Lidar range gate:', range_gates, 'm')

print('\nFile Configuration:')
for k in ['RANGE_GATE_LENGTH','SNR_THRESHOLD','CNS_PERCENTAGE']:
    print('-', k, ':', config_dict[k])

lon, lat, elev = float(config_dict['SYSTEM_LONGITUDE']), float(config_dict['SYSTEM_LATITUDE']), float(config_dict['SYSTEM_ALTITUDE'])

Lidar range gate: 18 m

File Configuration:
- RANGE_GATE_LENGTH : 18
- SNR_THRESHOLD : 0
- CNS_PERCENTAGE : 60


## Build datasets

### ICON

### RHI

In [8]:
ds_rhi_out = ds_rhi_in.copy(deep=True)

# Add global attributes
upd_attr = {
    "title": f"RHI scans from HALO StreamLine doppler lidar acquired during HEFEX III campaign",
    "institution": "Humboldt-Universität zu Berlin",
    "source": f"HALO StreamLine doppler lidar; processed with dl_toolbox (https://github.com/mkay-atm/dl_toolbox); see file configuration for details on processing settings",
    "instrument_mode": "RHI scan",
    "contact": "alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-6761)",
    "campaign": "HEFEX III",
    "location": f"HALO Streamline (lon, lat, elev): {lon:.6f}°E, {lat:.6f}°N, {elev:.2f}m",
    "StartTime": format_datetime64( ds_rhi_out.half_hour.values[0] ),
    "StopTime":  format_datetime64( ds_rhi_out.half_hour.values[-1] ),
    "comment": f"Manual post-processing includes 1° resampling, 30min averaging, computation of Cartesian coordinates x and z; Raw data available on request"
}

for k in ['instrument_id', 'data_policy', 'wigos_station_id', 'wmo_id']:
    del ds_rhi_out.attrs[k]

for k, v in upd_attr.items():
    ds_rhi_out.attrs[k] = v

# Write
output_file = data_out_dir + f'HEFEX3__Obs_RHI_L2__avg30min_{get_daterange_str(ds_rhi_out, time_dim='half_hour')}.nc'
ds_rhi_out.to_netcdf(output_file)
print(f'Output file saved as {output_file}.')

Output file saved as ../data_out/HEFEX3__Obs_RHI_L2__avg30min_20250806-20250817.nc.
